# 1. Explain and correct a classifier with SCE counterfactuals (CFKD)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Explainable-AI-Berlin/pytorch_explain_and_adapt_library/blob/master/notebooks/01_explain_and_correct_with_sce.ipynb)

Notebook 00 found a shortcut by sweeping a dictionary of *named* concepts (DiDAE). This notebook
uses PEAL's other route, **counterfactual explanations from a diffusion model**:

* **SCE** (*smooth counterfactual explorer*) takes an image and changes it, guided by the gradient of
  the classifier, until the classifier changes its decision, while a diffusion model keeps the image
  realistic and an L1 penalty keeps the change small. No dictionary, no concept names: the
  counterfactual shows *where* and *how* the image had to change.
* **CFKD** (*counterfactual knowledge distillation*) is the adaptor around it: it generates
  counterfactuals for many images, asks a **teacher** whether each one is a valid change of the class
  or a shortcut, and fine-tunes the classifier on the answers.

The building blocks are the same as in notebook 00 - configs and factories - with a different
generator (`DDPMConfig`), explainer (`SCEConfig`) and adaptor (`CFKDConfig`). Swapping components is
the point of PEAL: the model and data come from notebook 02 and stay unchanged.

**Hardware and runtime.** SCE backpropagates through the diffusion model, which makes it the most
demanding notebook. The pretrained ImageNet diffusion model is a 2.1 GB download (OpenAI's 256x256
unconditional model); the surrogate training takes a few minutes; then SCE explains the images one at
a time. On a 128 px diffusion model one image at a time needed 13.7 GB of GPU memory in our tests, so
plan for a large GPU (Colab: A100) with this 256 px model. The defaults explain 4 images. (We tested
every cell of this notebook with a 128 px CelebA diffusion model; the ImageNet model's weights could
not be downloaded on our test machine, so that path was not executed end to end.)


## 0. Setup


In [ ]:
# Setup: the same cell in every PEAL notebook. Works on Google Colab and locally.
# Colab: Runtime -> Change runtime type -> a GPU. The first run installs PEAL and
# restarts the runtime once (PEAL needs numpy < 2); then run this cell again.
import importlib.util, logging, os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PEAL_VERSION = "0.1.0"
PEAL_REPO = "https://github.com/Explainable-AI-Berlin/pytorch_explain_and_adapt_library.git"

if importlib.util.find_spec("peal") is None:
    # --ignore-requires-python: PEAL 0.1.0 declares Python < 3.12, Colab runs 3.12
    # (it works there). OpenAI's `clip` package is not on PyPI; the RAE generator needs it.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python",
                    f"peal-xai[rae,onnx]=={PEAL_VERSION}", "git+https://github.com/openai/CLIP.git"],
                   check=True)
    if IN_COLAB:
        print("PEAL installed. Restarting the runtime once; run this cell again afterwards.")
        os.kill(os.getpid(), 9)


def find_configs():
    """PEAL's config files live in its repository, not in the pip package.
    Use $PEAL_BASE, else the repository this notebook sits in, else a sparse clone
    (configs/ and the dictionary vocabulary only)."""
    if os.environ.get("PEAL_BASE"):
        return Path(os.environ["PEAL_BASE"])
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "configs" / "web_demo").is_dir():
            return parent
    target = Path("/content/peal" if IN_COLAB else Path.home() / "peal_configs")
    if not (target / "configs").is_dir():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse",
                        PEAL_REPO, str(target)], check=True)
        subprocess.run(["git", "-C", str(target), "sparse-checkout", "set", "configs",
                        "peal/dependencies/MSAE/vocab"], check=True)
    return target


os.environ["PEAL_BASE"] = str(find_configs())            # <PEAL_BASE> in config files
CONFIGS = Path(os.environ["PEAL_BASE"]) / "configs"
ROOT = Path("/content/peal_data") if IN_COLAB else Path.home() / "peal_data"
os.environ.setdefault("PEAL_DATA", str(ROOT / "datasets"))  # $PEAL_DATA: datasets
os.environ.setdefault("PEAL_RUNS", str(ROOT / "runs"))      # $PEAL_RUNS: models and runs
os.environ.setdefault("PEAL_RAE_WEIGHTS", "hf://sidney1505/peal-rae-clip-imagenet@v0.1.0")
DATA, RUNS = Path(os.environ["PEAL_DATA"]), Path(os.environ["PEAL_RUNS"])

import warnings
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")      # TensorFlow (pulled in by a dependency) is chatty
warnings.filterwarnings("ignore", message="To copy construct from a tensor")       # PEAL 0.1.0 dataset noise
warnings.filterwarnings("ignore", message='Field "model_', category=UserWarning)   # pydantic namespace notes
logging.getLogger("matplotlib.image").setLevel(logging.ERROR)                      # "Clipping input data" notes

import peal, shutil, torch
# PEAL 0.1.0's wheel misses the concept vocabulary of the MSAE dictionary; copy it in from the repo
from peal.sparse_dictionaries.msae_decomposition import MSAE_DIR
_vocab = Path(MSAE_DIR) / "vocab" / "clip_disect_20k.txt"
if not _vocab.exists():
    _vocab.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(Path(os.environ["PEAL_BASE"]) / "peal/dependencies/MSAE/vocab/clip_disect_20k.txt", _vocab)
from peal.log import configure
configure()   # PEAL logs progress at INFO; drop only its "Loading config ..." dumps and file lists
_quiet = lambda r: not str(r.getMessage()).startswith(
    ("Loading config", "Config model", "No config model", "Saved collage", "[DiDAE Sweep] decoded "))
for _h in logging.getLogger("peal").handlers:
    _h.addFilter(_quiet)
print(f"PEAL {peal.__version__} | configs: {CONFIGS}")
print(f"PEAL_DATA: {DATA} | PEAL_RUNS: {RUNS}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be very slow)")

# ---- notebook-specific -------------------------------------------------------------------
import io, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as PNG

WORK = RUNS / "sce_cfkd"
DEVICE = "cuda"

def show(fig):
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=90, bbox_inches="tight"); plt.close(fig)
    display(PNG(buf.getvalue()))


## 1. The model and data

### Option A (default): the ONNX freight-car classifier and data config from notebook 02

SCE follows the classifier's *gradient*. PEAL converts an ONNX graph into a differentiable torch
module (`onnx2torch`), so an ONNX model works here as well; a graph with an unsupported operator falls
back to inference-only onnxruntime and cannot be explained with SCE (use DiDAE, notebook 03).


In [ ]:
from peal.global_utils import load_yaml_config
from peal.data.interfaces import DataConfig

STUDENT = RUNS / "bring_your_own" / "freight_car_resnet18.onnx"
DATA_YAML = RUNS / "bring_your_own" / "data.yaml"
CLASS_NAMES = ["freight car", "passenger car"]
LABEL_COLUMN = "Class"
if not (STUDENT.exists() and DATA_YAML.exists()):
    raise FileNotFoundError("run notebook 02 first, or set STUDENT / DATA_YAML to your own files (Option B)")
data_config = load_yaml_config(str(DATA_YAML), DataConfig)


### Option B: your own model (`.onnx` or `.cpl`) and data config


In [ ]:
USE_OWN = False
if USE_OWN:
    STUDENT = Path("/path/to/my_model.onnx")
    data_config = load_yaml_config("/path/to/my_data.yaml", DataConfig)
    CLASS_NAMES, LABEL_COLUMN = ["class 0", "class 1"], "Class"
print("model:", STUDENT.name, "| images:", data_config.dataset_path, "| input", data_config.input_size)


## 2. The generator: `DDPMConfig`

A DDPM (denoising diffusion model) in PEAL is a `DDPMConfig`. The shipped `imagenet_ddpm.yaml`
describes OpenAI's pretrained 256x256 unconditional ImageNet model and downloads its weights on first
use (`download_weights`, `is_trained: true`): nothing to train. We point it at our images (in the
generator's own format: 256 px, pixels in [-1, 1]) and at a folder for the weights.

Images that look nothing like ImageNet (X-rays, microscopy, faces) need a DDPM trained on them:
`peal-train-generator --config <a DDPMConfig yaml>`, see `configs/sce_experiments/generators/`.


In [ ]:
from peal.generators.ddpm_generator import DDPMConfig

generator_config = load_yaml_config(str(CONFIGS / "sce_experiments/generators/imagenet_ddpm.yaml"), DDPMConfig)
generator_config.base_path = str(RUNS / "imagenet_ddpm256")          # the weights are downloaded here
generator_config.data = data_config.model_copy(update={"input_size": [3, 256, 256], "normalization": [0.5, 0.5]})
print("generator:", generator_config.generator_type, generator_config.image_size, "px | weights:",
      generator_config.download_weights)


## 3. The explainer: `SCEConfig`

The shipped `sce_default256x256.yaml` holds the settings of the SCE paper. The ones you are most
likely to touch:

* `gradient_steps` - optimisation steps per counterfactual (more: stronger, slower edits);
* `dist_l1` - weight of the L1 distance to the input (more: smaller, sparser changes);
* `inpaint` - how strongly unchanged regions are copied back from the input;
* `num_attempts` - SCE searches for this many *different* counterfactuals per image.

SCE first distils the classifier into a small, smooth surrogate (`distilled_predictor`, a short
training run on your images) whose gradients guide the diffusion model.


In [ ]:
import yaml
from peal.explainers.counterfactual_explainer import SCEConfig

explainer_config = load_yaml_config(str(CONFIGS / "sce_experiments/explainers/sce_default256x256.yaml"), SCEConfig)
print({k: getattr(explainer_config, k) for k in ("gradient_steps", "dist_l1", "inpaint", "num_attempts")})

# The surrogate is itself a config (a PredictorConfig, here as a dict): components are configs all the
# way down. The shipped one trains a ResNet-18 for 10 x 1000 steps. FAST_SURROGATE = True cuts that to
# 2 x 200 steps for a quick look; the gradients, and with them the counterfactuals, get worse.
FAST_SURROGATE = False
surrogate = yaml.safe_load(open(CONFIGS / "sce_experiments/predictors/latent_convexity_distillation_imagenet.yaml"))
surrogate["training"] = yaml.safe_load(open(CONFIGS / "sce_experiments/training/latent_convexity_distillation_imagenet.yaml"))
if FAST_SURROGATE:
    surrogate["training"].update(max_epochs=2, steps_per_epoch=200)
explainer_config.distilled_predictor = surrogate
print("surrogate:", surrogate["architecture"], "| training steps:",
      surrogate["training"]["max_epochs"] * surrogate["training"]["steps_per_epoch"])


## 4. The adaptor: `CFKDConfig` → `get_adaptor` → `run()`

CFKD runs in iterations: generate counterfactuals for a set of images, collect the teacher's verdicts,
fine-tune. `finetune_iterations=0` stops after the explanations, which is what we want first; the
teacher `"Baseline:true"` accepts every counterfactual without asking anybody.
`max_validation_samples` is how many images are explained.


In [ ]:
from peal.adaptors.counterfactual_knowledge_distillation import CFKDConfig
from peal.adaptors.adaptor_factory import get_adaptor
from peal.architectures.interfaces import TaskConfig

N_IMAGES = 4
cfkd_config = CFKDConfig(
    student=str(STUDENT),
    data=data_config, test_data=data_config,
    task=TaskConfig(criterions={"ce": 1.0, "l2": 100.0}, output_type="singleclass",
                    output_channels=2, y_selection=[LABEL_COLUMN]),
    generator=generator_config,
    explainer=explainer_config,
    teacher="Baseline:true",
    finetune_iterations=0,            # explain only
    max_validation_samples=N_IMAGES,  # how many images get counterfactuals
    batch_size=1,                     # SCE backpropagates through the diffusion model: memory-hungry
    base_dir=str(WORK),
    calculate_explainer_stats=False,
    tracking_level=3,                 # 3: write the counterfactuals to the run directory
    overwrite=True,
)
run_dir = Path(cfkd_config.base_dir) / "0"
if (run_dir / "validation_tracked_values.npz").exists():
    print("reusing", run_dir)
else:
    t0 = time.time(); torch.cuda.reset_peak_memory_stats()
    cfkd = get_adaptor(cfkd_config)
    cfkd.run()
    print(f"CFKD/SCE finished in {(time.time() - t0) / 60:.1f} min, "
          f"peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")
    del cfkd; torch.cuda.empty_cache()
print(sorted(p.name for p in run_dir.iterdir()))


## 5. The counterfactuals

`validation_tracked_values.npz` holds the explained images and their counterfactuals in matching order
(`x_list`, `x_counterfactual_list`) together with the classifier's confidence in the target class on
each counterfactual. Not every attempt crosses the decision boundary within `gradient_steps`; those are
shown in grey. The difference image shows where SCE changed the input.


In [ ]:
tracked = np.load(run_dir / "validation_tracked_values.npz", allow_pickle=True)
print("arrays:", sorted(tracked.files))
x = np.asarray(tracked["x_list"], dtype=np.float32)                 # pixels in [0, 1]
cf = np.asarray(tracked["x_counterfactual_list"], dtype=np.float32)
source, target = tracked["y_source_list"], tracked["y_target_list"]
confidence = tracked["y_target_end_confidence_list"]                # the classifier on the counterfactual
flipped = confidence > 0.5
print(f"{int(flipped.sum())} of {len(flipped)} counterfactuals reach the target class")

img = lambda a: np.clip(a.transpose(1, 2, 0), 0, 1)
show_idx = np.r_[np.flatnonzero(flipped), np.flatnonzero(~flipped)][:6]   # successful ones first
fig, axes = plt.subplots(3, len(show_idx), figsize=(2.6 * len(show_idx), 8), squeeze=False)
for col, i in enumerate(show_idx):
    axes[0, col].imshow(img(x[i])); axes[1, col].imshow(img(cf[i]))
    axes[2, col].imshow(np.abs(img(cf[i]) - img(x[i])).sum(-1), cmap="magma")
    axes[0, col].set_title(CLASS_NAMES[int(source[i])], fontsize=9)
    axes[1, col].set_title(f"→ {CLASS_NAMES[int(target[i])]} {confidence[i]:.2f}", fontsize=9,
                           color="black" if flipped[i] else "gray")
    for r in range(3):
        axes[r, col].axis("off")
axes[0, 0].set_ylabel("original"); axes[1, 0].set_ylabel("counterfactual"); axes[2, 0].set_ylabel("|difference|")
plt.tight_layout(); show(fig)


CFKD also writes ready-made collages: `validation_collages0_1/` holds the best counterfactual per
image (`..._0/` the second, different one SCE was forced to find), and `val_counterfactuals_global.png`
an overview.


In [ ]:
overview = run_dir / "val_counterfactuals_global.png"
if overview.exists():
    display(PNG(filename=str(overview)))


## 6. Correct the classifier with CFKD

With verdicts, CFKD becomes a correction method: counterfactuals the teacher calls **false** (the
classifier changed its mind for a reason that should not matter) become training data with the
*original* label, and the student is fine-tuned on them mixed with the training data
(`mixing_ratio`). The teacher can be

* `"human@8000"` - a labelling web page served on port 8000 (`"cluster@8000"` groups similar
  counterfactuals so you judge a few clusters instead of every image),
* the path of an **oracle** model (`.cpl`) that knows the true label,
* `"Baseline:true"` / `"Baseline:false"` - accept / reject everything (for testing).

The cell below is switched off: it needs a teacher that can tell shortcuts from class changes.


In [ ]:
CORRECT = False
if CORRECT:
    correct_config = cfkd_config.model_copy(update={
        "teacher": "human@8000",               # or "cluster@8000", or "/path/to/oracle/model.cpl"
        "finetune_iterations": 1,
        "base_dir": str(WORK / "corrected"),
    })
    cfkd = get_adaptor(correct_config)
    corrected_student = cfkd.run()


## Where next

* **Concept-level explanations instead of pixel edits:** notebooks 00 and 03 (DiDAE).
* **Other diffusion explainers:** DiME, ACE and FastDiME share this interface; their configs are in
  `configs/sce_experiments/explainers/` (`dime_default.yaml`, `ace_default.yaml`, ...).
* **From the command line:** `peal-cfkd --config <a CFKDConfig yaml>` runs the same adaptor; the
  shipped examples are under `configs/sce_experiments/adaptors/`.
